# Oxford RobotCar → ElideDB, step by step

This notebook builds the `lake/oxford` database **from the raw dataset**, one
step per cell, and ends with a cell that clears the datastore so you can
rebuild from scratch any time.

**Prerequisites:** `pip install -e ".[ml]"` from the repo root, `brew install
ffmpeg`, and the raw sample at `Data/oxfordDataset/` with this layout:

```
Data/oxfordDataset/
  stereo/{centre,left,right}/   1280×960 raw Bayer PNGs (filename = epoch µs)
  mono_{left,right,rear}/       1024×1024 raw Bayer PNGs
  gps/{gps.csv, ins.csv}        GPS ~5 Hz, INS 50 Hz (timestamp in µs)
  lms_front/, lms_rear/, ldmrs/ lidar scans as float64 .bin files
```

### Where the gigabyte goes (raw 1.04 GB → database ≈ 0.37 GB)

Nothing is dropped — every one of the 1,443 frames and every sensor row ends
up in the database. The size shrinks because the **representation** changes:

| raw form | stored form | why smaller |
|---|---|---|
| Bayer-mosaic PNGs (~250 KB/frame, lossless mosaic) | demosaiced JPEG q92 packed into one AVI per camera | JPEG on natural images ≈ 0.3× lossless mosaic |
| CSV text (gps/ins) | Parquet + zstd, columnar | ~10× smaller than text |
| float64 lidar scans | per-scan summary rows | v1 keeps summaries, not point clouds |

If you need bit-identical raw frames retained, keep the raw folder (ElideDB
never touches it) or ingest with `copy=False` to reference files in place.

In [ ]:
# ── 0 · setup ────────────────────────────────────────────────────────────
import sys, shutil, subprocess, tempfile
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "python"))

import cv2, numpy as np, pandas as pd
from elidedb import Store, cluster
from elidedb.fftools import find as find_tool

DATASET = Path("../Data/oxfordDataset")   # ← the raw dataset (read-only)
STORE   = Path("../lake/oxford")          # ← the database this notebook builds
assert DATASET.is_dir(), f"raw dataset not found at {DATASET.resolve()}"
print("raw dataset :", DATASET.resolve())
print("database at :", STORE.resolve())

## 1 · Create the database

A database is just a directory. If this cell says it already exists, either
keep using it or jump to the **clear** cell at the bottom first.

In [ ]:
if (STORE / "_store.json").exists():
    db = Store.open(STORE)
    print(f"store already exists — {len(db.tables())} tables. "
          "Run the CLEAR cell at the bottom first for a fresh build.")
else:
    db = Store.create(STORE, "oxford-robotcar-sample")
    print("created empty database:", db.name)

## 2 · Upload the six cameras

RobotCar cameras store **raw Bayer-mosaic** PNGs — one file per frame, the
filename is the capture time in epoch microseconds. The helper below:

1. demosaics each frame (stereo = GBRG pattern, mono = RGGB),
2. JPEG-encodes (q92) and packs all frames into one AVI per camera
   (a pile of small files has no efficient random-access story; one packed
   file + a byte-range index does),
3. hands ElideDB the AVI **plus the exact per-frame nanosecond timestamps**.

`copy=True` (the default) moves the packed AVI into the store's `media/`
directory, so the store stays a single self-contained folder.

In [ ]:
def pack_camera(cam_dir: Path, bayer_code: int, workdir: str):
    """raw Bayer PNGs -> (packed .avi path, per-frame ns timestamps)"""
    frames = sorted(cam_dir.glob("*.png"))
    raw = Path(workdir) / (cam_dir.name + ".mjpeg")
    ts_ns = []
    with open(raw, "wb") as out:
        for p in frames:
            img = cv2.cvtColor(cv2.imread(str(p), cv2.IMREAD_GRAYSCALE), bayer_code)
            ok, jpg = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 92])
            assert ok, p
            out.write(jpg.tobytes())
            ts_ns.append(int(p.stem) * 1000)          # µs filename -> ns
    avi = raw.with_suffix(".avi")                      # remux for exact packet
    subprocess.run([find_tool("ffmpeg"), "-v", "error", "-y",  # byte offsets
                    "-f", "mjpeg", "-i", str(raw), "-c:v", "copy", str(avi)],
                   check=True)
    raw.unlink()
    return avi, ts_ns

CAMERAS = {  # dir -> (bayer pattern, stream name)
    "stereo/centre": (cv2.COLOR_BayerGR2BGR, "stereo/centre"),
    "stereo/left":   (cv2.COLOR_BayerGR2BGR, "stereo/left"),
    "stereo/right":  (cv2.COLOR_BayerGR2BGR, "stereo/right"),
    "mono_left":     (cv2.COLOR_BayerBG2BGR, "mono/left"),
    "mono_right":    (cv2.COLOR_BayerBG2BGR, "mono/right"),
    "mono_rear":     (cv2.COLOR_BayerBG2BGR, "mono/rear"),
}

with tempfile.TemporaryDirectory() as wd:
    for rel, (code_, stream) in CAMERAS.items():
        avi, ts_ns = pack_camera(DATASET / rel, code_, wd)
        v = db.ingest_video("frames", avi, timestamps_ns=ts_ns, stream=stream)
        print(f"  {stream:<15} {len(ts_ns):>4} frames  -> frames table v{v}")
print("cameras done — media now lives inside", STORE / "media")

## 3 · Upload GPS and INS

Straight from the raw CSVs — no preprocessing. `ts_column` names the
dataset's timestamp column; the µs epoch unit is auto-detected. String
columns (`utm_zone`, `ins_status`) are kept as-is: any column type rides
along, only the timestamp is mandatory.

In [ ]:
for name in ("gps", "ins"):
    v = db.ingest_rows(name, DATASET / "gps" / f"{name}.csv", ts_column="timestamp")
    print(f"  {name}: {db.table(name).state().rows:,} rows -> v{v}")

## 4 · Upload lidar activity

The `.bin` scans are float64 arrays — `lms_*` are 2-D (x, y, reflectance),
`ldmrs` is 3-D (x, y, z). We store one summary row per scan (count +
min/mean/max range) so lidar activity is queryable and alignable like any
other sensor. (Full point-cloud storage would just be another table with
more columns.)

In [ ]:
def lidar_summary(scan_dir: Path, dims: int) -> pd.DataFrame:
    rows = []
    for p in sorted(scan_dir.glob("*.bin")):
        a = np.fromfile(p, np.float64).reshape(3, -1)
        r = np.hypot(a[0], a[1]) if dims == 2 else np.sqrt((a ** 2).sum(0))
        rows.append({"ts": int(p.stem) * 1000, "n_points": r.size,
                     "mean_range": r.mean(), "min_range": r.min(),
                     "max_range": r.max()})
    return pd.DataFrame(rows)

for name, dims in (("lms_front", 2), ("lms_rear", 2), ("ldmrs", 3)):
    v = db.ingest_rows(name, lidar_summary(DATASET / name, dims), ts_unit="ns")
    print(f"  {name}: {db.table(name).state().rows:,} scans -> v{v}")

## 5 · Verify what you uploaded

In [ ]:
import pandas as pd
inv = pd.DataFrame(db.describe())[["table", "kind", "rows", "bytes", "files", "version"]]
media = sum(p.stat().st_size for p in (STORE / "media").glob("*"))
total = sum(p.stat().st_size for p in STORE.rglob("*") if p.is_file())
print(f"database size: {total/1e9:.2f} GB standalone "
      f"(media {media/1e9:.2f} GB, tables {(total-media)/1e6:.1f} MB)")
inv

## 6 · Embed for semantic search

Local SigLIP (first run downloads the model, ~2 GB, then cached). Windows
are the *indexing* granularity — search results merge them into dynamic
segments — so 1 s is a good default here. Re-run this cell whenever you add
more video.

In [ ]:
print(db.embed_windows(window_s=1.0, frames_per_window=2))
print(cluster(db, min_cluster_size=5))

## 7 · First query

In [ ]:
import matplotlib.pyplot as plt
hits, stats = db.search_text("pedestrians walking on the sidewalk", k=4)
print(stats)
fig, axes = plt.subplots(1, len(hits), figsize=(16, 3.4))
for ax, h in zip(axes, hits):
    w, _ = db.window(h["t0"], h["t1"], tables=["frames"])
    dec = w["frames"].decode(stream=h["stream"], width=400, limit=1)
    if dec:
        ax.imshow(dec[0][1])
    ax.axis("off")
    ax.set_title(f"{h['score']:.3f} · {h['stream']} · "
                 f"{(h['t1']-h['t0'])/1e9:.1f}s", fontsize=9)
plt.tight_layout()

Browse it visually: `elidedb desk` (or the macOS app) — Overview,
Timeline, **Storage** (the Parquet files and row groups), the semantic map,
and click-to-play clips.

---

## ⚠ Clear the datastore

Deletes `lake/oxford` **entirely** (database only — the raw dataset under
`Data/` is never touched). Set `CONFIRM = True` and run.

In [ ]:
CONFIRM = False   # ← set to True, then run this cell

if CONFIRM:
    import shutil
    shutil.rmtree(STORE, ignore_errors=True)
    print("cleared:", STORE.resolve(),
          "— re-run this notebook from the top to rebuild.")
else:
    print("not cleared — set CONFIRM = True first.")